In [37]:
# Please specify the csv file path here
data_path = "./data/WineQuality.csv"

In [38]:
# Static threshold variables used in code
unique_ratio_threshold_id_column = 0.95
# low_cardinality_threshold1 = 0.1
# low_cardinality_threshold2 = 0.3
target_keywords = ["target", "label", "class", "output", "result", "status", "y", "outcome"]
unique_values_threshold_for_classification = 50
high_cardinality_threshold = 20
correlation_threshold = 0.8
missing_info_percent_threshold = 40
imbalance_ratio_threshold = 0.7
dataset_size_threshold = 500
datetime_column_conversion_threshold = 0.8
categorical_column_unique_ratio_threshold = 0.05
categorical_column_value_range_threshold = 20

In [39]:
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import json

Determine Ground-truth label and Problem type 

In [40]:
# ID Column Detection
import warnings


def detect_id_columns(df):
    id_columns = []

    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]) or pd.api.types.is_bool_dtype(df[col]):
            continue
        unique_ratio = df[col].nunique() / len(df)
        # if it has 'id' in the name
        if "id" in col.lower():
            id_columns.append(col)
        # if high unique ratio and non numeric
        elif unique_ratio > unique_ratio_threshold_id_column and not pd.api.types.is_numeric_dtype(df[col]):
            id_columns.append(col)
        # if high unique ratio and increasing monotonically 
        elif unique_ratio > unique_ratio_threshold_id_column and df[col].is_monotonic_increasing:
            id_columns.append(col)
        # if numeric and constant step size 
        elif pd.api.types.is_numeric_dtype(df[col]) and df[col].diff().dropna().nunique() == 1:
            id_columns.append(col)
        elif df[col].dtype == 'object':
            avg_length = df[col].astype(str).str.len().mean()
            # likely hash/uuid
            if avg_length > 15 and unique_ratio > unique_ratio_threshold_id_column:
                id_columns.append(col)
    return list(set(id_columns))

# Target Candidate Scoring
def score_target_candidates(df, id_columns):

    scores = {}

    for col in df.columns:
        score = 0
        # unique_ratio = df[col].nunique() / len(df)

        # ---- Signal 0: Low Cardinality ----
        # if unique_ratio < low_cardinality_threshold1:
        #     score += 3
        # elif unique_ratio < low_cardinality_threshold2:
        #     score += 1

        # Signal 1: id column
        if col in id_columns:
            score -= 4  

        #  Signal 2: Column Name 
        if col.lower() in target_keywords:
            score += 4

        #  Signal 3: Last Column Bias 
        if col == df.columns[-1]:
            score += 2

        #  Signal 4: Datatype Pattern 
        if not pd.api.types.is_numeric_dtype(df[col]) and not pd.api.types.is_bool_dtype(df[col]) and col not in id_columns:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=UserWarning)
                converted_to_datetime = pd.to_datetime(df[col], errors="coerce")
                
            n_valid_dates = converted_to_datetime.notna().sum()
            if (n_valid_dates / len(col)) >= datetime_column_conversion_threshold:
                score -= 4
                
        scores[col] = score

    return scores

# Select Best Candidate
def detect_target(df):

    id_columns = detect_id_columns(df)
    scores = score_target_candidates(df, id_columns)

    if len(scores) == 0:
        return None, scores

    predicted_target = max(scores, key=scores.get)

    if scores[predicted_target] == 0:
        return None, scores

    return predicted_target, scores

# Problem Type Detection (Connected)
def detect_problem_type(df, target):

    if target is None:
        return "Unsupervised"

    unique_values = df[target].nunique()

    if not pd.api.types.is_numeric_dtype(df[target]) or unique_values < unique_values_threshold_for_classification:
        return "Classification"

    return "Regression"

def explain_target_choice(target, scores):

    print("\nTarget Detection Report")
    print("---------------------------------------------------------------------------------------------")

    if target is None or len(scores) == 0:
        print("No reliable target detected. Dataset treated as unsupervised.")
        return

    import math

    exp_scores = {
        col: math.exp(score)
        for col, score in scores.items()
    }

    total = sum(exp_scores.values())

    percentage_scores = {
        col: (val / total) * 100
        for col, val in exp_scores.items()
    }

    sorted_scores = sorted(
        percentage_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    print(f"Predicted Target Column: {target}")
    print("\n-----Target Probability Ranking-----")

    for col, percent in sorted_scores:
        print(f"{col}: {percent:.2f}%")
    print("\n")


Signal Extraction

In [41]:
import warnings


def extract_dataset_overview(df):
    return {
        "dataset_size": {"rows": int(df.shape[0]), "cols": int(df.shape[1])},
        # "num_features": df.shape[1],
        "duplicate_count": df.duplicated().sum()
    }

def behaves_like_categorical(df, col):
    series = df[col].dropna()
    if len(series) == 0:
        return False
    unique_ratio = series.nunique() / len(series)
    value_range = series.max() - series.min()
    is_integer = pd.api.types.is_integer_dtype(series)
    if unique_ratio < categorical_column_unique_ratio_threshold and is_integer and value_range < categorical_column_value_range_threshold:
        return True
    return False

def extract_feature_types(df):

    # numerical = df.select_dtypes(include=["number"]).columns.tolist()
    # categorical = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    # datetime = df.select_dtypes(include=["datetime64[ns]", "datetime64"]).columns.tolist()
    id_cols = detect_id_columns(df)

    # Detect numeric columns that behave like categorical
    # for col in numerical.copy():
    #     unique_ratio = df[col].nunique() / len(df)
    #     print(col, ": ", unique_ratio)
    #     if unique_ratio < 0.05:  # heuristic threshold
    #         numerical.remove(col)
    #         categorical.append(col)

    # for col in categorical.copy():
    #     if col in id_cols:
    #         categorical.remove(col)
    schema = {}
    for column in df.columns:
        if column in id_cols:
            continue
        col = df[column]
        
        if pd.api.types.is_numeric_dtype(col):
            # n_unique = col.nunique()
            # n_rows = len(col)
            # threshold = max(10, 0.05 * n_rows)
            # if n_unique <= threshold:
            #     schema[column] = "categorical"
            # else:
            #     schema[column] = "numeric"
            if behaves_like_categorical(df, column):
                schema[column] = "categorical"
            else:
                schema[column] = "numeric"
        
        elif pd.api.types.is_datetime64_any_dtype(col):
            schema[column] = "datetime"
        
        elif pd.api.types.is_bool_dtype(col):
            schema[column] = "categorical"
        
        else:
            
            # Attempt to convert to datetime
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=UserWarning)
                converted_to_datetime = pd.to_datetime(col, errors="coerce")
                
            n_valid_dates = converted_to_datetime.notna().sum()
            if n_valid_dates / len(col) >= datetime_column_conversion_threshold:
                schema[column] = "datetime"
            else:
                n_unique = col.nunique()
                n_rows = len(col)
                threshold = max(10, categorical_column_unique_ratio_threshold * n_rows)
                if n_unique <= threshold:
                    schema[column] = "categorical"
                else:
                    schema[column] = "text"

    # return {
    #     "numerical_features": numerical,
    #     "categorical_features": categorical,
    #     "datetime_features": datetime,
    #     "id_features": id_cols
    # }

    return {
        "numerical_features": [col for col, dtype in schema.items() if dtype == "numeric"],
        "categorical_features": [col for col, dtype in schema.items() if dtype == "categorical"],
        "datetime_features": [col for col, dtype in schema.items() if dtype == "datetime"],
        "text_features" : [col for col, dtype in schema.items() if dtype == "text"],
        "id_features": id_cols
    }

# Because later your planner can reason like:
# If global_missing_severity < 5%
#   → Minor issue → Simple imputation
# If 5%–20%
#   → Moderate issue → Careful strategy needed
# If > 30%
#   → Severe → Possibly drop columns
def extract_missing_signals(df):
    missing_percent = (df.isnull().sum() / len(df)) * 100

    return {
        "missing_percent": missing_percent.to_dict(),
        "columns_with_missing": missing_percent[missing_percent > 0].index.tolist(),
        "global_missing_severity": missing_percent.mean()
    }

def extract_target_signals(df):
    target, scores = detect_target(df)
    explain_target_choice(target, scores)

    if target is None:
        return {
            "target_column": None,
            "problem_type": "Unsupervised",
            "target_cardinality": None
        }

    problem_type = detect_problem_type(df, target)

    return {
        "target_column": target,
        "problem_type": problem_type,
        "target_cardinality": df[target].nunique()
    }

def extract_imbalance_signals(df, target):
    if target is None:
        return {
            "class_distribution": None,
            "imbalance_ratio": None
        }

    distribution = df[target].value_counts(normalize=True)
    
    return {
        "class_distribution": distribution.to_dict(),
        "imbalance_ratio": distribution.max()
    }

def extract_cardinality_signals(df, categorical_cols):
    high_card_cols = []

    for col in categorical_cols:
        if df[col].nunique() > high_cardinality_threshold:
            high_card_cols.append(col)

    return {
        "high_cardinality": high_card_cols
    }

def extract_skewness_signals(df, numerical_cols, threshold=1.0):    
    if not numerical_cols:
        return {
            "skewness_values": {},
            "highly_skewed_features": [],
            "moderately_skewed_features": []
        }
    
    skewness = df[numerical_cols].skew()

    highly_skewed = skewness[abs(skewness) > threshold].index.tolist()
    moderately_skewed = skewness[
        (abs(skewness) > 0.5) & (abs(skewness) <= threshold)
    ].index.tolist()

    return {
        "skewness_values": skewness.to_dict(),
        "highly_skewed_features": highly_skewed,
        "moderately_skewed_features": moderately_skewed
    }


def extract_outlier_signals(df, numerical_cols):
    outlier_cols = []

    for col in numerical_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]

        if len(outliers) > 0:
            outlier_cols.append(col)

    return {
        "outlier_columns": outlier_cols
    }

def extract_multicollinearity(df, numerical_cols):
    corr = df[numerical_cols].corr().abs()

    high_corr_pairs = []

    for i in range(len(corr.columns)):
        for j in range(i):
            if corr.iloc[i, j] > correlation_threshold:
                pair = (corr.columns[i], corr.columns[j])
                high_corr_pairs.append(pair)

    return {
        "highly_correlated_pairs": high_corr_pairs
    }


def extract_dataset_signals(df):
    signals = {}

    # Overview
    signals.update(extract_dataset_overview(df))

    # Feature types
    feature_signals = extract_feature_types(df)
    signals.update(feature_signals)

    # Missing values
    signals.update(extract_missing_signals(df))

    # Target detection
    target_signals = extract_target_signals(df)
    signals.update(target_signals)

    # Imbalance
    signals.update(
        extract_imbalance_signals(df, signals["target_column"])
    )

    # Cardinality
    signals.update(
        extract_cardinality_signals(df, signals["categorical_features"])
    )

    # Skewness
    signals.update(
        extract_skewness_signals(df, signals["numerical_features"])
    )

    # Outliers
    signals.update(
        extract_outlier_signals(df, signals["numerical_features"])
    )

    # Multicollinearity
    signals.update(
        extract_multicollinearity(df, signals["numerical_features"])
    )

    return signals


EDA

In [42]:

def generate_basic_statistics(df):

    print("\n---- BASIC DATASET STATISTICS ----")
    display(df.describe(include='all'))


def plot_numerical_distributions(df, numerical_cols):

    if len(numerical_cols) == 0:
        return

    df[numerical_cols].hist(figsize=(12, 8))
    plt.suptitle("Numerical Feature Distributions")
    plt.show()


def plot_categorical_distributions(df, categorical_cols):

    for col in categorical_cols:
        plt.figure()
        df[col].value_counts().head(10).plot(kind="bar")
        plt.title(f"Top Categories: {col}")
        plt.show()


def classification_target_analysis(df, target):

    print("\n---- TARGET DISTRIBUTION ----")
    df[target].value_counts().plot(kind="bar")
    plt.title("Class Distribution")
    plt.show()


def regression_target_analysis(df, target):

    plt.figure()
    df[target].hist()
    plt.title("Target Distribution")
    plt.show()


def classification_feature_relationship(df, numerical_cols, target):

    for col in numerical_cols:
        plt.figure()
        df.boxplot(column=col, by=target)
        plt.title(f"{col} vs {target}")
        plt.show()


def regression_feature_relationship(df, numerical_cols, target):

    for col in numerical_cols:
        plt.figure()
        plt.scatter(df[col], df[target])
        plt.title(f"{col} vs {target}")
        plt.xlabel(col)
        plt.ylabel(target)
        plt.show()


def correlation_analysis(df, numerical_cols):

    if len(numerical_cols) < 2:
        return

    corr = df[numerical_cols].corr()

    plt.figure(figsize=(8,6))
    plt.imshow(corr)
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.title("Correlation Matrix")
    plt.show()


def generate_adaptive_eda(df, signals):
    print("\n\nExploratory Data Analysis")
    print("---------------------------------------------------------------------------------------------")

    numerical_cols = signals["numerical_features"]
    categorical_cols = signals["categorical_features"]
    target = signals["target_column"]
    problem_type = signals["problem_type"]

    # Always run
    generate_basic_statistics(df)
    plot_numerical_distributions(df, numerical_cols)
    plot_categorical_distributions(df, categorical_cols)
    correlation_analysis(df, numerical_cols)

    # Conditional EDA
    if target is not None:
        if problem_type == "Classification":
            classification_target_analysis(df, target)
            classification_feature_relationship(df, numerical_cols, target)
        elif problem_type == "Regression":
            regression_target_analysis(df, target)
            regression_feature_relationship(df, numerical_cols, target)

Decision Engine

In [43]:
def action_decision_engine(signals):
    actions = {
        "eda": [],
        "preprocessing": [],
        "risks": [],
        "modelling": []
    }
    problem_type = signals["problem_type"]
    
    # Problem Type Based Decisions
    if problem_type == "Classification":
        actions["eda"].append("Plot target class distribution")
        actions["eda"].append("Analyse feature vs target relationships")
        actions["modelling"].append("Use classification models")
    elif problem_type == "Regression":
        actions["eda"].append("Analyse correlation between features and target")
        actions["eda"].append("Plot scatter plots of features vs target")
        actions["modelling"].append("Use regression models")
    else:
        actions["eda"].append("Analyse feature distributions")
        actions["eda"].append("Perform correlation analysis")
        actions["modelling"].append("Consider clustering or dimensionality reduction")

    
    # Missing Value Decisions
    missing_info = signals["missing_percent"]
    for col, pct in missing_info.items():
        if pct > missing_info_percent_threshold:
            actions["risks"].append(f"{col} has very high missing values")
            actions["preprocessing"].append(f"Consider dropping or advanced imputation for {col}")
        elif pct > 0:
            actions["preprocessing"].append(f"Apply imputation for {col}")

    
    # Imbalance Detection
    if signals["imbalance_ratio"] is not None:
        if signals["imbalance_ratio"] > imbalance_ratio_threshold:
            actions["risks"].append("Severe class imbalance detected")
            actions["modelling"].append("Use F1-score or balanced metrics")
            actions["modelling"].append("Use stratified sampling")

    
    # Dataset Size Risk
    if signals["dataset_size"]["rows"] < dataset_size_threshold:
        actions["risks"].append("Small dataset - risk of overfitting")
        actions["modelling"].append("Prefer simpler models")

    
    # Cardinality Detection
    high_card_cols = signals["high_cardinality"]
    
    for col in high_card_cols:
        actions["risks"].append(f"{col} has high cardinality")
        actions["preprocessing"].append(f"Use appropriate encoding for {col}")

    #Skewness detection
    highly_skewed = signals["highly_skewed_features"]
    moderately_skewed = signals["moderately_skewed_features"]
    if highly_skewed:
        actions["risks"].append(f"{moderately_skewed} are highly skewed features")
        actions["preprocessing"].append(f"Apply log or Box-Cox transformation to highly skewed features: {highly_skewed}")
        if problem_type == "regression":
            actions.append(
                "Skewness may affect linear regression performance."
            )
    if moderately_skewed:
        actions["risks"].append(f"{moderately_skewed} are moderately skewed features")
        actions["preprocessing"].append(f"Consider transformation for moderately skewed features: {moderately_skewed}")

    # Duplicate Handling
    if signals["duplicate_count"] > 0:
        actions["preprocessing"].append("Remove duplicate rows")

    return actions

def print_agent_knowledge(actions):
    
    print("\n\nKnowledg Report")
    print("---------------------------------------------------------------------------------------------")
    
    print("EDA Recommendations:")
    for item in actions["eda"]:
        print("-", item)
    
    print("\nDetected Risks:")
    for item in actions["risks"]:
        print("-", item)

    print("\nPreprocessing Recommendations:")
    for item in actions["preprocessing"]:
        print("-", item)
    
    print("\nModelling Strategy Suggestions:")
    for item in actions["modelling"]:
        print("-", item)

In [44]:
def numpy_converter(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    return str(obj)

In [45]:
df = pd.read_csv(data_path)

signals = extract_dataset_signals(df)

print("Dataset signals")
print("---------------------------------------------------------------------------------------------")
print(json.dumps(signals, indent=2, ensure_ascii=False, default=numpy_converter))

# generate_adaptive_eda(df, signals)

actions = action_decision_engine(signals)
print_agent_knowledge(actions)


Target Detection Report
---------------------------------------------------------------------------------------------
Predicted Target Column: quality

-----Target Probability Ranking-----
quality: 40.18%
fixed acidity: 5.44%
volatile acidity: 5.44%
citric acid: 5.44%
residual sugar: 5.44%
chlorides: 5.44%
free sulfur dioxide: 5.44%
total sulfur dioxide: 5.44%
density: 5.44%
pH: 5.44%
sulphates: 5.44%
alcohol: 5.44%


Dataset signals
---------------------------------------------------------------------------------------------
{
  "dataset_size": {
    "rows": 1699,
    "cols": 12
  },
  "duplicate_count": 240,
  "numerical_features": [
    "fixed acidity",
    "volatile acidity",
    "citric acid",
    "residual sugar",
    "chlorides",
    "free sulfur dioxide",
    "total sulfur dioxide",
    "density",
    "pH",
    "sulphates",
    "alcohol",
    "quality"
  ],
  "categorical_features": [],
  "datetime_features": [],
  "text_features": [],
  "id_features": [],
  "missing_percent":